# For RQ1: Built environment v. car ownership

**Question (RQ part 1)**
- Is mixed-use development linked to lower car ownership?
- How does it compare to density, street connectivity, transit access?

**Data**
- EPA Smart Location Database v3.0
- 1 row = 1 Census block group
- cleaned in `src/build_sld.py`

**Sample**
- metros with 1M+ people ("major cities")
- block groups with 50+ households

**Outcome**
- `pct_2plus_cars` = % of households with 2+ cars
- averages weighted by # of households

In [7]:
import numpy as np
import pandas as pd
import altair as alt

sld = pd.read_csv("../data/processed/sld_blockgroups.csv.gz", dtype={"geoid": str, "cbsa": str})

# keep major metros + block groups with enough households
df = sld[sld["major_metro"] & sld["enough_households"]].copy()

print(len(df), "block groups")
print(df["cbsa"].nunique(), "metros")
df[["pct_2plus_cars", "mixed_use", "pop_density", "intersection_density", "transit_dist_m"]].describe().round(2)

114860 block groups
54 metros


,pct_2plus_cars,mixed_use,pop_density,intersection_density,transit_dist_m
count,114860.00,114860.00,114860.00,114860.00,72504.00
mean,55.27,0.48,16.68,101.39,444.36
std,23.63,0.23,36.76,83.84,275.91
min,0.00,0.00,0.00,0.00,0.00
25%,38.98,0.31,3.52,44.00,237.38
50%,58.46,0.50,7.95,86.96,387.58
75%,74.23,0.66,15.58,138.48,607.53
max,100.00,1.00,3133.58,1930.42,1207.00


## Step 1 · Put features on the same scale

- 4 features, 4 different units → can't compare directly
- fix: turn each into a **percentile** (0 = most car-oriented, 100 = most urban)
- split into 10 bins → avg % with 2+ cars in each bin
- steeper line = stronger link to car ownership

Transit note
- SLD has no distance if nearest stop > ¾ mile
- those all tied at "worst access" → end up as one point

In [8]:
# transit: closer = better, so flip the sign
df["transit_access"] = -df["transit_dist_m"]
# no stop within 3/4 mile = worst possible
df.loc[df["no_stop_within_3_4mi"], "transit_access"] = -np.inf

# column name -> chart label
FEATURES = {
    "mixed_use": "Mixed use",
    "pop_density": "Population density",
    "intersection_density": "Street connectivity",
    "transit_access": "Transit access",
}
OUTCOME = "pct_2plus_cars"


def profile(data, col, label):
    """Avg % with 2+ cars (household-weighted) in each of 10 percentile bins."""
    d = data[[col, OUTCOME, "households"]].dropna().copy()

    # percentile (0-100) and bin (1-10)
    d["pct"] = d[col].rank(pct=True) * 100
    d["bin"] = np.ceil(d["pct"] / 10).clip(1, 10).astype(int)

    # weighted avg = sum(outcome * households) / sum(households)
    d["cars_x_hh"] = d[OUTCOME] * d["households"]
    g = d.groupby("bin").agg(
        x=("pct", "mean"),
        cars_x_hh=("cars_x_hh", "sum"),
        hh=("households", "sum"),
        n=("households", "size"),
    ).reset_index()
    g["y"] = (g["cars_x_hh"] / g["hh"]).round(1)
    g["feature"] = label
    return g[["feature", "bin", "x", "y", "n"]]


# one profile per feature, stacked into one table
tables = []
for col, label in FEATURES.items():
    tables.append(profile(df, col, label))
prof = pd.concat(tables, ignore_index=True)
prof.round(1)

,feature,bin,x,y,n
0,Mixed use,1,5.0,50.0,11486
1,Mixed use,2,15.0,57.9,11486
2,Mixed use,3,25.0,59.6,11486
3,Mixed use,4,35.0,59.2,11486
4,Mixed use,5,45.0,57.8,11486
5,Mixed use,6,55.0,56.7,11486
6,Mixed use,7,65.0,55.9,11486
7,Mixed use,8,75.0,55.5,11486
8,Mixed use,9,85.0,54.4,11486
9,Mixed use,10,95.0,52.0,11486


## Step 2 · Figure 1: which feature tracks car ownership most?

- x = percentile on each feature
- y = % of households with 2+ cars
- 1 line per feature

In [9]:
# pick colors; mixed use = red since it's our focus
colors = alt.Scale(
    domain=["Mixed use", "Population density", "Street connectivity", "Transit access"],
    range=["#d1495b", "#3a6ea5", "#2a9d8f", "#8d6cab"],
)

fig1 = alt.Chart(prof).mark_line(point=True).encode(
    x=alt.X("x:Q", title="Percentile (0 = most car-oriented, 100 = most urban)"),
    y=alt.Y("y:Q", title="Households with 2+ cars (%)", scale=alt.Scale(zero=False)),
    color=alt.Color("feature:N", title="Feature", scale=colors),
    tooltip=["feature", "y", "n"],
).properties(
    width=500,
    height=320,
    title="Which neighborhood features track car ownership most closely?",
)

fig1.save("../figures/fig1_feature_comparison.png", scale_factor=2)
fig1.save("../figures/fig1_feature_comparison.html")
fig1

alt.Chart(...)

**Numbers for write-up**
- `gap` = drop in % with 2+ cars, least urban bin → most urban bin
- `spearman` = rank correlation w/ % with 2+ cars (more negative = stronger)

In [10]:
# first bin = least urban, last bin = most urban
summary = prof.sort_values("bin").groupby("feature").agg(
    least_urban=("y", "first"),
    most_urban=("y", "last"),
)
summary["gap"] = summary["least_urban"] - summary["most_urban"]

for col, label in FEATURES.items():
    summary.loc[label, "spearman"] = df[col].corr(df[OUTCOME], method="spearman")

summary.sort_values("gap", ascending=False).round(2)

,least_urban,most_urban,gap,spearman
feature,,,,
Population density,73.3,21.5,51.8,-0.48
Transit access,68.3,32.2,36.1,-0.52
Street connectivity,68.6,37.4,31.2,-0.36
Mixed use,50.0,52.0,-2.0,-0.02


## Step 3 · Figure 2: does mixed use still matter at similar densities?

- problem: dense places are also mixed, connected, near transit
- → Fig 1 can't separate mixed use from density
- fix: split into 3 density groups (low / medium / high)
- redo the profiles **inside** each group
- line still steep → feature matters even at similar density
- line flat → Fig 1 slope was mostly just density

In [11]:
# 3 equal-size density groups
df["density_group"] = pd.qcut(df["pop_density"], 3, labels=["Low density", "Medium density", "High density"])

tables = []
for group_name in ["Low density", "Medium density", "High density"]:
    group = df[df["density_group"] == group_name]
    for col, label in FEATURES.items():
        if col == "pop_density":
            continue  # skip density itself
        t = profile(group, col, label)
        t["density_group"] = group_name
        tables.append(t)
prof2 = pd.concat(tables, ignore_index=True)

fig2 = alt.Chart(prof2).mark_line(point=True).encode(
    x=alt.X("x:Q", title="Percentile within density group"),
    y=alt.Y("y:Q", title="Households with 2+ cars (%)", scale=alt.Scale(zero=False)),
    color=alt.Color("density_group:N", title="Density group",
                    sort=["Low density", "Medium density", "High density"],
                    scale=alt.Scale(range=["#9ecae1", "#4292c6", "#08306b"])),  # light -> dark blue
    column=alt.Column("feature:N", title=None),
    tooltip=["feature", "density_group", "y", "n"],
).properties(
    width=200,
    height=240,
    title="Holding density fixed, which features still track car ownership?",
)

fig2.save("../figures/fig2_within_density.png", scale_factor=2)
fig2.save("../figures/fig2_within_density.html")
fig2

alt.Chart(...)

In [12]:
# gap (least urban - most urban bin) for each feature x density group
gaps = prof2.sort_values("bin").groupby(["feature", "density_group"]).agg(
    least_urban=("y", "first"),
    most_urban=("y", "last"),
)
gaps["gap"] = gaps["least_urban"] - gaps["most_urban"]

# rows = feature, columns = density group
gaps.reset_index().pivot(index="feature", columns="density_group", values="gap")[
    ["Low density", "Medium density", "High density"]
].round(1)

density_group,Low density,Medium density,High density
feature,,,
Mixed use,7.8,6.5,5.8
Street connectivity,25.1,11.8,-4.1
Transit access,20.7,19.1,29.8


## Next steps (not needed for proposal)

- **Income:** redo Fig 2 split by income (ACS median HH income, join on `geoid`)
- **Commuting:** swap outcome for ACS drive-alone / transit / walk shares (table B08301)
- **CO₂ (RQ part 2):** join Vulcan on-road emissions → plot per resident + per km² vs. same percentiles